# Gaussian Process Toy Example

In this example, we will recreate the simple toy model from [Rasmussen & Williams (2006)](https://doi.org/10.1198/jasa.2008.s219) Section 2.2 to demonstrate how a Gaussian process (GP) works. In this example, we will use data both with and without uncertainty on the dependent variable to show how uncertainty affects the GP.

In [ ]:
import torch
import gpytorch
import matplotlib.pyplot as plt
_ = torch.manual_seed(123)

First, let's create a small data set and plot it.

In [ ]:
train_x = torch.tensor([-4, -2, -0.5, 1, 3])
train_fx = torch.tensor([-2.1, -0.8, 0.9, 2, -1.4])
train_fx_unc = torch.abs(train_fx) * 0.03

fig, ax = plt.subplots()
_ = ax.errorbar(train_x, train_fx, train_fx_unc,
                capsize=3,color='black', linestyle='',
                marker='o', markersize=4)
_ = ax.set_xlim([-5, 5])
_ = ax.set_ylim([-2.5, 2.5])
_ = ax.set_xlabel('input, x')
_ = ax.set_ylabel('output, f(x)')

Next, let's make our GP model. For this example, we will use a process with a zero mean and a kernel function of a squared exponential (SE).

In [ ]:
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        # RBF is a SE kernel in gpytorch. Set the lengthscale to be 1.
        self.covar_module = gpytorch.kernels.RBFKernel(
            lengthscale_constraint=gpytorch.constraints.Interval(0.99999, 1.00001)
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

To train the model and make predictions, let's create a few helper functions as we will need to train and predict twice.

In [ ]:
def train(model, likelihood, training_iter=500):
    # Use the adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(training_iter):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model(train_x)
        # Calc loss and backprop gradients
        loss = -mll(output, train_fx)
        loss.backward()
        optimizer.step()


test_x = torch.linspace(-5, 5, 201)
def predict(model, likelihood, test_x=test_x):
    model.eval()
    likelihood.eval()
    # Make predictions by feeding model through likelihood
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        return likelihood(model(test_x))

def sample(model, likelihood, test_x=test_x, n=3):
    model.eval()
    likelihood.eval()
    # sample from the model
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        f_preds = model(test_x)
        sim = f_preds.sample(sample_shape=torch.Size([n]))
        return sim

Now, we can create the models, one without and one with uncertainty. Additionally, we will make a model to show the prior of the model.

In [ ]:
likelihood_wo_unc = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=torch.zeros(5))
likelihood_w_unc = gpytorch.likelihoods.FixedNoiseGaussianLikelihood(noise=train_fx_unc)

# To get the prior model, feed in blank tensors and any likelihood
model_prior = ExactGPModel(torch.tensor([]), torch.tensor([]), likelihood_wo_unc)
model_wo_unc = ExactGPModel(train_x, train_fx, likelihood_wo_unc)
model_w_unc = ExactGPModel(train_x, train_fx, likelihood_w_unc)

Finally, we can make some plots to see how the results compare. To do this we need to make some predictions and draw a few samples from the posterior predictive distribution.

> Note that we do not need to train the models, since we are not tuning any hyperparameters (i.e., we are fixing the lengthscale to 1).

In [ ]:
_ = torch.manual_seed(123)
models = {'prior': model_prior, 'wo_unc': model_wo_unc, 'w_unc': model_w_unc}
likelihoods = {'prior': likelihood_wo_unc,
               'wo_unc': likelihood_wo_unc,
               'w_unc': likelihood_w_unc}

# Initialize plot
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)

for model_type, ax in zip(models, axes):
    prediction = predict(models[model_type], likelihoods[model_type])
    sim = sample(models[model_type], likelihoods[model_type])

    with torch.no_grad():
        # Get upper and lower confidence bounds
        lower, upper = prediction.confidence_region()
    
        # Shade between the lower and upper confidence bounds
        confid = ax.fill_between(test_x.numpy(), lower.numpy(), upper.numpy(), alpha=0.3, color='gray', edgecolor=None)

        # Plot sampled posterior functions
        samp = ax.plot(test_x.numpy(), sim.T.numpy())
 
        # Plot predictive means
        pred = ax.plot(test_x.numpy(), prediction.mean.numpy(), 'black', linestyle='--')
        
        # Plot training data
        if model_type == 'wo_unc':
            ax.plot(train_x.numpy(), train_fx.numpy(), 'ko', markersize=4)
            ax.set_title('(b) GP Posterior without Uncertainty')
        elif model_type == 'w_unc':
            data = ax.errorbar(train_x, train_fx, train_fx_unc,
                        capsize=3, capthick=2, color='black', linestyle='',
                        marker='o', markersize=4)
            ax.set_title('(c) GP Posterior with Uncertainty')
        elif model_type == 'prior':
            ax.set_ylabel('output, f(x)', fontsize=12)
            ax.set_title('(a) GP Prior')

        ax.set_xlabel('input, x', fontsize=12)
        ax.set_xlim([-5, 5])
        ax.tick_params(axis="both", direction="in", bottom=True, top=True, right=True, left=True)

  
ax.set_ylim([-4, 4])
from matplotlib.legend_handler import HandlerTuple, HandlerLineCollection

fig.legend([confid, pred[0], tuple(samp), data],
           ['Confidence', 'Mean', 'Function Draws', 'Observed Data'],
           handler_map={tuple: HandlerTuple(ndivide=None)},
           loc=[0.06, 0.75], ncols=2)
plt.tight_layout()